### Redocrd Grouping

In [ ]:
"""
Exact 160×5 Grouping Script with JSON Output
This script loads `combined_80multi_unique.json`, computes full cluster similarities, 
selects exactly 160 groups of 5 clusters with 200 leftovers, and writes the result as a JSON file.
"""

from __future__ import annotations

import json
import math
import re
from collections import Counter
from dataclasses import dataclass
from itertools import combinations
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import csc_array, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import os
from pathlib import Path

project_root = Path("").resolve()
os.chdir(project_root)

# Paths
input_path = "./dataset/WDC_80multi_unique.json"
output_json = "./dataset/WDC-grouping.json"

# Candidate generation / optimization parameters
neighbor_pool = 10
top_per_seed = 8
time_limit = 300
random_seed = 42

@dataclass(frozen=True)
class CandidateGroup:
    members: Tuple[int, ...]  # sorted cluster positions
    score: float
    origin: str


def normalize_text(value: object) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and math.isnan(value):
        return ""
    text = str(value).lower()
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def summarize_values(values: Iterable[object], top_k: int, max_len: int) -> List[str]:
    counter: Counter[str] = Counter()
    for value in values:
        text = normalize_text(value)
        if text:
            counter[text] += 1
    return [text[:max_len] for text, _ in counter.most_common(top_k)]


def build_cluster_table(df: pd.DataFrame) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for cluster_id, g in df.groupby("cluster_id", sort=True):
        brands = summarize_values(g.get("brand", []), top_k=3, max_len=60)
        titles = summarize_values(g.get("title", []), top_k=6, max_len=180)
        descs = summarize_values(g.get("description", []), top_k=2, max_len=240)

        rep_parts: List[str] = []
        rep_parts.extend([f"brand {x}" for x in brands])
        rep_parts.extend([f"title {x}" for x in titles])
        rep_parts.extend([f"desc {x}" for x in descs])

        rows.append(
            {
                "cluster_id": int(cluster_id),
                "offer_count": int(len(g)),
                "brands": brands,
                "titles": titles,
                "descriptions": descs,
                "cluster_text": " ".join(rep_parts),
            }
        )

    cluster_df = pd.DataFrame(rows).reset_index(drop=True)
    cluster_df["cluster_pos"] = np.arange(len(cluster_df))
    return cluster_df


def build_similarity_matrix(cluster_texts: List[str]) -> np.ndarray:
    word_vec = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        max_features=30000,
        sublinear_tf=True,
    )
    char_vec = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        max_features=40000,
        sublinear_tf=True,
    )

    x_word = word_vec.fit_transform(cluster_texts)
    x_char = char_vec.fit_transform(cluster_texts)
    x = hstack([x_word, x_char]).tocsr()

    sim = linear_kernel(x, x)
    np.fill_diagonal(sim, 1.0)
    return np.asarray(sim, dtype=np.float64)


def group_score(sim: np.ndarray, members: Tuple[int, ...]) -> float:
    vals = [sim[a, b] for a, b in combinations(members, 2)]
    return float(np.mean(vals)) if vals else 0.0


def group_stats(sim: np.ndarray, members: Tuple[int, ...]) -> Tuple[float, float, float]:
    vals = [float(sim[a, b]) for a, b in combinations(members, 2)]
    if not vals:
        return 0.0, 0.0, 0.0
    return float(np.mean(vals)), float(np.min(vals)), float(np.max(vals))


def medoid_position(sim: np.ndarray, members: Tuple[int, ...]) -> int:
    best_pos = members[0]
    best_score = -1.0
    for p in members:
        s = 0.0
        for q in members:
            if p != q:
                s += float(sim[p, q])
        s /= max(1, len(members) - 1)
        if s > best_score:
            best_score = s
            best_pos = p
    return best_pos


def generate_topk_candidates(
    sim: np.ndarray,
    neighbor_pool: int = 10,
    top_per_seed: int = 8,
) -> List[CandidateGroup]:
    n = sim.shape[0]
    seen: Dict[Tuple[int, ...], CandidateGroup] = {}

    for seed in range(n):
        order = np.argsort(-sim[seed])
        neigh = [int(p) for p in order if int(p) != seed][:neighbor_pool]

        scored: List[Tuple[float, Tuple[int, ...]]] = []
        for combo in combinations(neigh, 4):
            members = tuple(sorted((seed, *combo)))
            scored.append((group_score(sim, members), members))

        scored.sort(reverse=True)
        for score, members in scored[:top_per_seed]:
            prev = seen.get(members)
            cand = CandidateGroup(members=members, score=score, origin="topk")
            if prev is None or cand.score > prev.score:
                seen[members] = cand

    return list(seen.values())


def greedy_partition(
    sim: np.ndarray,
    positions: List[int],
    strategy: str,
    rng: np.random.Generator,
) -> List[CandidateGroup]:
    remaining = set(int(p) for p in positions)
    groups: List[CandidateGroup] = []

    def pick_seed() -> int:
        rem = sorted(remaining)

        if strategy == "random":
            return int(rng.choice(rem))

        if strategy == "avg_local":
            best_seed = rem[0]
            best_val = -1.0
            for p in rem:
                others = [q for q in rem if q != p]
                if not others:
                    continue
                top = sorted((sim[p, q] for q in others), reverse=True)[:8]
                val = float(np.mean(top)) if top else -1.0
                if val > best_val:
                    best_val = val
                    best_seed = p
            return best_seed

        return max(
            rem,
            key=lambda p: float(np.mean([sim[p, q] for q in rem if q != p][:20] or [0.0])),
        )

    while len(remaining) >= 5:
        seed = pick_seed()
        rem_others = [q for q in remaining if q != seed]
        rem_others.sort(key=lambda q: sim[seed, q], reverse=True)

        chosen = tuple(sorted((seed, *rem_others[:4])))
        groups.append(
            CandidateGroup(
                members=chosen,
                score=group_score(sim, chosen),
                origin=f"partition_{strategy}",
            )
        )
        for p in chosen:
            remaining.remove(p)

    return groups


def build_feasible_partition_candidates(sim: np.ndarray, seed: int = 42) -> List[CandidateGroup]:
    rng = np.random.default_rng(seed)
    positions = list(range(sim.shape[0]))
    cands: Dict[Tuple[int, ...], CandidateGroup] = {}

    for strategy in ["avg_local", "degree", "random", "random", "random"]:
        groups = greedy_partition(sim, positions, strategy, rng)
        for cand in groups:
            prev = cands.get(cand.members)
            if prev is None or cand.score > prev.score:
                cands[cand.members] = cand

    return list(cands.values())


def dedupe_candidates(cands: List[CandidateGroup]) -> List[CandidateGroup]:
    best: Dict[Tuple[int, ...], CandidateGroup] = {}
    for c in cands:
        prev = best.get(c.members)
        if prev is None or c.score > prev.score:
            best[c.members] = c
    return list(best.values())


def solve_exact_group_selection(
    cands: List[CandidateGroup],
    cluster_count: int,
    target_groups: int = 160,
    time_limit: int = 300,
):
    m = len(cands)
    row_idx: List[int] = []
    col_idx: List[int] = []
    data: List[float] = []

    for j, cand in enumerate(cands):
        for p in cand.members:
            row_idx.append(p)
            col_idx.append(j)
            data.append(1.0)

    A1 = csc_array((data, (row_idx, col_idx)), shape=(cluster_count, m))
    b_l1 = np.zeros(cluster_count)
    b_u1 = np.ones(cluster_count)

    A2 = csc_array(np.ones((1, m), dtype=float))
    b_l2 = np.array([float(target_groups)])
    b_u2 = np.array([float(target_groups)])

    A = csc_array(np.vstack([A1.toarray(), A2.toarray()]))
    constraints = LinearConstraint(
        A,
        np.concatenate([b_l1, b_l2]),
        np.concatenate([b_u1, b_u2]),
    )

    bounds = Bounds(np.zeros(m), np.ones(m))
    integrality = np.ones(m, dtype=int)
    c = -np.array([cand.score for cand in cands], dtype=float)

    res = milp(
        c=c,
        constraints=constraints,
        integrality=integrality,
        bounds=bounds,
        options={
            "time_limit": float(time_limit),
            "presolve": True,
            "mip_rel_gap": 0.0,
        },
    )
    return res


def build_output_json(
    selected: List[CandidateGroup],
    cluster_df: pd.DataFrame,
    sim: np.ndarray,
) -> Dict[str, object]:
    selected_sorted = sorted(selected, key=lambda g: g.score, reverse=True)

    groups_json: List[Dict[str, object]] = []
    used_positions: set[int] = set()

    for rank, group in enumerate(selected_sorted, start=1):
        gid = f"G{rank:03d}"
        mean_sim, min_sim, max_sim = group_stats(sim, group.members)
        seed_pos = medoid_position(sim, group.members)
        seed_cluster_id = int(cluster_df.loc[seed_pos, "cluster_id"])

        ordered_members = [seed_pos] + [p for p in group.members if p != seed_pos]
        group_members_json: List[Dict[str, object]] = []

        for member_order, pos in enumerate(ordered_members, start=1):
            used_positions.add(pos)
            cluster_id = int(cluster_df.loc[pos, "cluster_id"])
            sim_to_seed = 1.0 if pos == seed_pos else float(sim[seed_pos, pos])

            group_members_json.append(
                {
                    "cluster_id": cluster_id,
                    "member_order_in_group": member_order,
                    "is_seed": bool(pos == seed_pos),
                    "similarity_to_seed": sim_to_seed,
                    "offer_count": int(cluster_df.loc[pos, "offer_count"]),
                    "brands": cluster_df.loc[pos, "brands"],
                    "titles": cluster_df.loc[pos, "titles"],
                }
            )

        groups_json.append(
            {
                "group_id": gid,
                "group_rank_desc": rank,
                "group_origin": group.origin,
                "seed_cluster_id": seed_cluster_id,
                "group_size": 5,
                "mean_pairwise_similarity": mean_sim,
                "min_pairwise_similarity": min_sim,
                "max_pairwise_similarity": max_sim,
                "clusters": group_members_json,
            }
        )

    leftover_positions = [
        int(p)
        for p in cluster_df["cluster_pos"].tolist()
        if int(p) not in used_positions
    ]

    leftovers_json: List[Dict[str, object]] = []
    for idx, pos in enumerate(leftover_positions, start=1):
        leftovers_json.append(
            {
                "leftover_rank": idx,
                "cluster_id": int(cluster_df.loc[pos, "cluster_id"]),
                "offer_count": int(cluster_df.loc[pos, "offer_count"]),
                "brands": cluster_df.loc[pos, "brands"],
                "titles": cluster_df.loc[pos, "titles"],
            }
        )

    assert len(groups_json) == 160, f"Expected 160 groups, got {len(groups_json)}"
    assert len(leftovers_json) == 200, f"Expected 200 leftovers, got {len(leftovers_json)}"

    return {
        "summary": {
            "total_clusters": int(len(cluster_df)),
            "group_count": int(len(groups_json)),
            "clusters_in_groups": int(len(groups_json) * 5),
            "leftover_count": int(len(leftovers_json)),
        },
        "groups": groups_json,
        "leftovers": leftovers_json,
    }


if __name__ == "__main__":
    with open(input_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    df = pd.DataFrame(raw)
    print("Columns:", list(df.columns))
    print("Rows:", len(df))
    print("Unique cluster_id:", df["cluster_id"].nunique())

    cluster_df = build_cluster_table(df)
    print("Cluster table size:", cluster_df.shape)

    sim = build_similarity_matrix(cluster_df["cluster_text"].tolist())
    print("Similarity matrix shape:", sim.shape)

    topk_cands = generate_topk_candidates(
        sim,
        neighbor_pool=neighbor_pool,
        top_per_seed=top_per_seed,
    )
    partition_cands = build_feasible_partition_candidates(sim, seed=random_seed)
    candidates = dedupe_candidates(topk_cands + partition_cands)

    print("Top-k candidates:", len(topk_cands))
    print("Partition candidates:", len(partition_cands))
    print("Deduplicated candidates:", len(candidates))

    res = solve_exact_group_selection(
        candidates,
        cluster_count=len(cluster_df),
        target_groups=160,
        time_limit=time_limit,
    )

    print("Success:", getattr(res, "success", False))
    print("Status:", getattr(res, "status", None))
    print("Message:", getattr(res, "message", ""))

    if not getattr(res, "success", False):
        raise RuntimeError(f"MILP did not solve successfully. status={res.status}, message={res.message}")

    x = np.rint(res.x).astype(int)
    selected = [cand for cand, flag in zip(candidates, x) if flag == 1]

    print("Selected groups:", len(selected))

    output = build_output_json(selected, cluster_df, sim)

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print(f"Wrote JSON to: {output_json}")
    print(output["summary"])

    # Quick inspection
    with open(output_json, "r", encoding="utf-8") as f:
        result = json.load(f)

    print(result["summary"])
    print("First group id:", result["groups"][0]["group_id"])
    print("First leftover cluster:", result["leftovers"][0]["cluster_id"])

### Sample Generation

In [ ]:
"""
Data Sampling Script
Handles matching groups to the original dataset, selecting extra clusters,
and generating stratified sample CSV datasets.
"""

import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd


# ============================================================
# Paths
# ============================================================

input_json_path = "./dataset/WDC_80multi_unique.json"
recovered_json_path = "./dataset/WDC_grouping.json"

sample1_csv_path = "./dataset/sample_dataset_1.csv"
sample2_csv_path = "./dataset/sample_dataset_2.csv"
manifest_path = "./dataset/WDC_sample_dataset_manifest.json"

RANDOM_SEED = 20
rng = random.Random(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# ============================================================
# Load original dataset
# ============================================================
with open(input_json_path, "r", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame(raw)

required_cols = {"id", "cluster_id"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in original dataset: {missing}")

cluster_size_map = df.groupby("cluster_id").size().to_dict()


# ============================================================
# Load top 32 groups from recovered JSON
# ============================================================
def load_top_groups_from_json(recovered_json_path, top_n=32):
    recovered_json_path = Path(recovered_json_path)

    if not recovered_json_path.exists():
        raise FileNotFoundError(f"Recovered JSON not found: {recovered_json_path}")

    with open(recovered_json_path, "r", encoding="utf-8") as f:
        obj = json.load(f)

    if "groups" not in obj:
        raise ValueError("Recovered JSON does not contain key 'groups'.")

    groups = []
    for g in obj["groups"][:top_n]:
        groups.append({
            "group_id": g["group_id"],
            "rank_desc": g.get("group_rank_desc", len(groups) + 1),
            "cluster_ids": [x["cluster_id"] for x in g["clusters"]],
        })

    return groups


top_groups = load_top_groups_from_json(recovered_json_path, top_n=32)

if len(top_groups) != 32:
    raise ValueError(f"Expected 32 top groups, found {len(top_groups)}")


# ============================================================
# Exact size features for groups
# ============================================================
for g in top_groups:
    g["cluster_sizes"] = [int(cluster_size_map[cid]) for cid in g["cluster_ids"]]
    g["sorted_sizes"] = sorted(g["cluster_sizes"])


def group_distance(g1, g2):
    """
    Distance between two groups based on exact sorted cluster sizes.
    Lower = more similar.
    """
    a = np.array(g1["sorted_sizes"], dtype=float)
    b = np.array(g2["sorted_sizes"], dtype=float)
    return float(np.sum(np.abs(a - b)))


def greedy_pair_groups_exact(groups):
    remaining = groups.copy()
    pairs = []

    # sort by total size then lexicographically by exact size profile
    remaining.sort(key=lambda g: (sum(g["sorted_sizes"]), g["sorted_sizes"]), reverse=True)

    while remaining:
        g = remaining.pop(0)

        best_idx = None
        best_dist = None

        for i, h in enumerate(remaining):
            d = group_distance(g, h)
            if best_dist is None or d < best_dist:
                best_dist = d
                best_idx = i

        mate = remaining.pop(best_idx)
        pairs.append((g, mate))

    return pairs


pairs = greedy_pair_groups_exact(top_groups)

sample1_groups = []
sample2_groups = []

for g1, g2 in pairs:
    # random orientation within matched pair
    if rng.random() < 0.5:
        sample1_groups.append(g1)
        sample2_groups.append(g2)
    else:
        sample1_groups.append(g2)
        sample2_groups.append(g1)

assert len(sample1_groups) == 16
assert len(sample2_groups) == 16


def flatten_group_clusters(groups):
    out = []
    for g in groups:
        out.extend(g["cluster_ids"])
    return out


sample1_group_cluster_ids = flatten_group_clusters(sample1_groups)
sample2_group_cluster_ids = flatten_group_clusters(sample2_groups)

sample1_group_sizes = [int(cluster_size_map[cid]) for cid in sample1_group_cluster_ids]
sample2_group_sizes = [int(cluster_size_map[cid]) for cid in sample2_group_cluster_ids]

print("Exact group size counts:")
print("Sample 1:", Counter(sample1_group_sizes))
print("Sample 2:", Counter(sample2_group_sizes))


# ============================================================
# Extra clusters:
# choose 20 + 20 from remaining clusters such that:
# 1) sample1 extras and sample2 extras have similar exact sizes
# 2) extras are close to the size distribution in the selected groups
# ============================================================
used_cluster_ids = set(sample1_group_cluster_ids) | set(sample2_group_cluster_ids)
all_cluster_ids = set(df["cluster_id"].unique().tolist())
remaining_cluster_ids = sorted(all_cluster_ids - used_cluster_ids)

remaining_records = [
    {"cluster_id": int(cid), "size": int(cluster_size_map[cid])}
    for cid in remaining_cluster_ids
]

# target size multiset for one sample's 20 extras:
# sample proportional to the exact sizes appearing in all selected groups
all_selected_group_sizes = sample1_group_sizes + sample2_group_sizes
target_size_pool = all_selected_group_sizes.copy()

# if target pool has fewer than 20 unique draws, sampling with replacement is fine
target_extra_sizes = sorted(rng.choices(target_size_pool, k=20))

print("Target exact extra sizes:", target_extra_sizes)


def select_40_candidates_closest_to_target(remaining_records, target_sizes, k_total=40):
    """
    Greedily choose 40 clusters from remaining pool so their sizes are close
    to a doubled target (because we need 20 extras for each of 2 samples).
    """
    doubled_target = sorted(target_sizes + target_sizes)
    pool = remaining_records.copy()

    selected = []

    for t in doubled_target:
        if not pool:
            break

        best_idx = None
        best_dist = None

        for i, rec in enumerate(pool):
            d = abs(rec["size"] - t)
            if best_dist is None or d < best_dist:
                best_dist = d
                best_idx = i

        selected.append(pool.pop(best_idx))

    # if still short, fill by smallest remaining size deviation from overall mean
    while len(selected) < k_total and pool:
        overall_mean = np.mean(target_sizes)
        best_idx = min(range(len(pool)), key=lambda i: abs(pool[i]["size"] - overall_mean))
        selected.append(pool.pop(best_idx))

    if len(selected) != k_total:
        raise ValueError(f"Expected {k_total} selected extra candidates, got {len(selected)}")

    return selected


extra_pool_40 = select_40_candidates_closest_to_target(
    remaining_records=remaining_records,
    target_sizes=target_extra_sizes,
    k_total=40
)

# sort by exact size and pair neighbors
extra_pool_40 = sorted(extra_pool_40, key=lambda x: (x["size"], x["cluster_id"]))

extra1 = []
extra2 = []

for i in range(0, 40, 2):
    a = extra_pool_40[i]
    b = extra_pool_40[i + 1]

    # assign pair across samples so exact-size multisets stay similar
    if rng.random() < 0.5:
        extra1.append(a)
        extra2.append(b)
    else:
        extra1.append(b)
        extra2.append(a)

assert len(extra1) == 20
assert len(extra2) == 20

extra1_cluster_ids = [x["cluster_id"] for x in extra1]
extra2_cluster_ids = [x["cluster_id"] for x in extra2]

assert set(extra1_cluster_ids).isdisjoint(set(extra2_cluster_ids))

extra1_sizes = [x["size"] for x in extra1]
extra2_sizes = [x["size"] for x in extra2]

print("Exact extra size counts:")
print("Sample 1 extras:", Counter(extra1_sizes))
print("Sample 2 extras:", Counter(extra2_sizes))


# ============================================================
# Final sample cluster sets
# ============================================================
sample1_cluster_ids = set(sample1_group_cluster_ids) | set(extra1_cluster_ids)
sample2_cluster_ids = set(sample2_group_cluster_ids) | set(extra2_cluster_ids)

assert len(sample1_cluster_ids) == 100
assert len(sample2_cluster_ids) == 100
assert sample1_cluster_ids.isdisjoint(sample2_cluster_ids)

sample1_df = df[df["cluster_id"].isin(sample1_cluster_ids)].copy()
sample2_df = df[df["cluster_id"].isin(sample2_cluster_ids)].copy()

# preserve original columns
sample1_df = sample1_df[df.columns.tolist()]
sample2_df = sample2_df[df.columns.tolist()]

sample1_df.to_csv(sample1_csv_path, index=False)
sample2_df.to_csv(sample2_csv_path, index=False)


# ============================================================
# Diagnostics
# ============================================================
def exact_size_distribution_distance(sizes1, sizes2):
    """
    L1 distance between exact size-count dictionaries.
    Lower = more similar.
    """
    c1 = Counter(sizes1)
    c2 = Counter(sizes2)
    keys = sorted(set(c1) | set(c2))
    return sum(abs(c1[k] - c2[k]) for k in keys)

group_dist_distance = exact_size_distribution_distance(sample1_group_sizes, sample2_group_sizes)
extra_dist_distance = exact_size_distribution_distance(extra1_sizes, extra2_sizes)
group_extra_distance_1 = exact_size_distribution_distance(sample1_group_sizes, extra1_sizes)
group_extra_distance_2 = exact_size_distribution_distance(sample2_group_sizes, extra2_sizes)

print("Distribution distances:")
print("Groups: sample1 vs sample2 =", group_dist_distance)
print("Extras: sample1 vs sample2 =", extra_dist_distance)
print("sample1: groups vs extras =", group_extra_distance_1)
print("sample2: groups vs extras =", group_extra_distance_2)


# ============================================================
# Manifest
# ============================================================
def manifest_group_entry(g):
    return {
        "group_id": g["group_id"],
        "rank_desc": g["rank_desc"],
        "cluster_ids": g["cluster_ids"],
        "cluster_sizes": g["cluster_sizes"],
        "sorted_sizes": g["sorted_sizes"],
    }

manifest = {
    "random_seed": RANDOM_SEED,
    "sample_1": {
        "groups": [manifest_group_entry(g) for g in sample1_groups],
        "extra_clusters": extra1,
        "group_size_counter": dict(Counter(sample1_group_sizes)),
        "extra_size_counter": dict(Counter(extra1_sizes)),
        "total_clusters": 100,
        "total_records": int(len(sample1_df)),
    },
    "sample_2": {
        "groups": [manifest_group_entry(g) for g in sample2_groups],
        "extra_clusters": extra2,
        "group_size_counter": dict(Counter(sample2_group_sizes)),
        "extra_size_counter": dict(Counter(extra2_sizes)),
        "total_clusters": 100,
        "total_records": int(len(sample2_df)),
    },
    "distance_summary": {
        "group_distribution_distance": int(group_dist_distance),
        "extra_distribution_distance": int(extra_dist_distance),
        "sample1_group_vs_extra_distance": int(group_extra_distance_1),
        "sample2_group_vs_extra_distance": int(group_extra_distance_2),
    }
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(f"Saved: {sample1_csv_path}")
print(f"Saved: {sample2_csv_path}")
print(f"Saved: {manifest_path}")

### Ground Truth Generation

In [ ]:
"""
Ground Truth Generation Script
Takes the sample datasets and creates adjacent positive pairs within clusters.
"""

import pandas as pd
from pathlib import Path

# ----------------------------
# Paths
# ----------------------------

sample1_path = "./dataset/sample_dataset_1.csv"
sample2_path = "./dataset/sample_dataset_2.csv"

gt1_path = "./dataset/sample_dataset_1_gt.csv"
gt2_path = "./dataset/sample_dataset_2_gt.csv"

# ----------------------------
# Function: build ground truth
# ----------------------------
def generate_ground_truth_adjacent_pairs(sample_csv_path, output_csv_path):
    """
    Generate ground truth in the same format as your provided GT file:
    columns = ['ltable_id', 'rtable_id']

    Rule:
    - group by cluster_id
    - preserve row order inside each cluster
    - create adjacent positive pairs only:
      (id1,id2), (id2,id3), ..., (id_{n-1}, id_n)
    """

    df = pd.read_csv(sample_csv_path)

    required_cols = {"id", "cluster_id"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {sample_csv_path.name}: {missing}")

    gt_rows = []

    # keep original row order
    df = df.reset_index(drop=False).rename(columns={"index": "_row_order"})

    for cluster_id, group in df.groupby("cluster_id", sort=False):
        group = group.sort_values("_row_order")
        ids = group["id"].tolist()

        if len(ids) >= 2:
            for i in range(len(ids) - 1):
                gt_rows.append({
                    "ltable_id": ids[i],
                    "rtable_id": ids[i + 1]
                })

    gt_df = pd.DataFrame(gt_rows, columns=["ltable_id", "rtable_id"])
    gt_df.to_csv(output_csv_path, index=False)

    print(f"Saved: {output_csv_path}")
    print(f"Rows: {len(gt_df)}")
    return gt_df

# ----------------------------
# Run for sample 1 and 2
# ----------------------------
if __name__ == "__main__":
    gt1 = generate_ground_truth_adjacent_pairs(sample1_path, gt1_path)
    gt2 = generate_ground_truth_adjacent_pairs(sample2_path, gt2_path)

    # ----------------------------
    # Quick preview
    # ----------------------------
    print("\nSample 1 GT preview:")
    print(gt1.head())

    print("\nSample 2 GT preview:")
    print(gt2.head())